In [365]:
with open('input.txt', 'r') as f:
    text = f.read()



In [366]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [367]:
chars = sorted(list(set(text)))
print(chars)
print(len(chars))

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
65


In [368]:
stoi = { c:i for i,c in enumerate(chars)}
itos = { i:c for i,c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
print(encode("hello"))
print(decode(encode("hello")))

[46, 43, 50, 50, 53]
hello


In [369]:
import torch 
train_data = text[:int(0.9*len(text))]
val_data = text[int(0.9*len(text)):]

encode_train = torch.tensor(encode(train_data), dtype=torch.long)
encode_val = torch.tensor(encode(val_data), dtype=torch.long)

print(train_data[:100])
print(encode_train[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [370]:
torch.manual_seed(1337)

block_size = 8
rand_int = torch.randint(0, len(encode_train) - block_size, (1,))

x_test = encode_train[rand_int:rand_int+block_size].tolist()
y_test = encode_train[rand_int+1:rand_int+block_size+1].tolist()

print(x_test)
print(y_test)
print(decode(x_test))
print(decode(y_test))

x_batch = torch.stack([encode_train[i:i+block_size] for i in rand_int])
y_batch = torch.stack([encode_train[i+1:i+block_size+1] for i in rand_int])

def sample_batch( batch_size, train = True):
    if train:
        rand_int = torch.randint(0, len(encode_train) - block_size, (batch_size,))
    else:
        rand_int = torch.randint(0, len(encode_val) - block_size, (batch_size,))
    x = torch.stack([encode_train[i:i+block_size] for i in rand_int])
    y = torch.stack([encode_train[i+1:i+block_size+1] for i in rand_int])
    
    return x, y
        



[24, 43, 58, 5, 57, 1, 46, 43]
[43, 58, 5, 57, 1, 46, 43, 39]
Let's he
et's hea


In [371]:
import torch.nn as nn
import torch.nn.functional as F
class BigramGPT(nn.Module):
    def __init__(self, vocab_size, n_embd, block_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd) # T,C 
        self.lm_head = nn.Linear(n_embd, vocab_size)
    def __call__(self, idx):
        self.out = self.embedding(idx) # B,T C
        self.out += self.position_embedding(torch.arange(self.out.shape[1])) # B,T,C
        self.out = self.lm_head(self.out) # B,T,vocab_size
        return self.out
    
    def generate(self, idx, max_new_tokens):
        # idx will be a B,T tensor
        
        # x will be a B,T,C tensor
        for _ in range(max_new_tokens):
            x = self(idx)
            x = x [ :, -1, :]
            probs = F.softmax(x, dim = -1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim = 1)
        return idx
            

class Attention(nn.Module):
    def __init__(self, n_embd, block_size, head_size):
        super().__init__()
        self.n_embd = n_embd
        self.block_size = block_size
        self.mask = torch.tril(torch.ones(block_size,block_size)) # T,T 
        self.mask[self.mask == 0] = float('-inf')
        self.mask = F.softmax(self.mask, dim = -1)
        
        self.wk = nn.Linear(n_embd, head_size)
        self.wq = nn.Linear(n_embd, head_size)
        self.wv = nn.Linear(n_embd, head_size)
        self.block_size = block_size
    def __call__(self, x):
        k = self.wk(x)
        q = self.wq(x)
        v = self.wv(x)
        kq = q @ k.transpose(1,2)
        kq = kq / torch.sqrt(torch.tensor(self.block_size))
        kq = kq.masked_fill(self.mask == 0, float('-inf'))
        kq = F.softmax(kq, dim = -1)
        self.out = kq @ v
        return self.out

class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, block_size, head_size, n_heads, dropout = 0.2):
        super().__init__()
        self.n_embd = n_embd
        self.block_size = block_size
        self.head_size = head_size
        self.n_heads = n_heads
        self.attentions = nn.ModuleList([Attention(n_embd, block_size, head_size) for _ in range(n_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)
    
    def __call__(self, x):
        out = torch.cat([attention(x) for attention in self.attentions], dim = -1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

class FeedForward(nn.Module):
    def __init__(self, n_embd, out_dim, dropout = 0.2):
        super().__init__()
        self.n_embd = n_embd
        self.out_dim = out_dim
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 *n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, out_dim),
            nn.Dropout(dropout),
        )
      
    def forward(self, x):
        return self.net(x)
    
class Block(nn.Module):
    def __init__(self, n_embd, block_size, n_heads):
        super().__init__()
        head_size = n_embd//n_heads
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.attention = MultiHeadAttention(n_embd, block_size, head_size, n_heads)
        self.feed_forward = FeedForward(n_embd, n_embd)
    
    def forward(self, x):
        x = self.ln1(x)
        x = self.attention(x) + x
        x = self.ln2(x)
        x = self.feed_forward(x) + x
        return x

class Transformer(nn.Module):
    def __init__(self, vocab_size, n_embd, block_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd) # T,C 
        
        self.blocks = nn.ModuleList([Block(n_embd, block_size, n_heads=4) for _ in range(4)])
        self.ln = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.block_size = block_size

    def __call__(self, idx):
        x = self.embedding(idx) # B,T C
        x += self.position_embedding(torch.arange(self.block_size)) # B,T,C
        for block in self.blocks:
            x = block(x)
        x = self.ln(x)
        self.out = self.lm_head(x)
        
        return self.out
    



In [372]:




epochs = 10000
batch_size = 32
vocab_size = len(chars)
n_embed = 32

model = Transformer(vocab_size, n_embed, block_size)
import torch.optim as optim
optimizer = optim.AdamW(model.parameters(), lr=1e-2)


In [373]:
for i in range(epochs):
    x,y = sample_batch(batch_size, train = True)
    out = model(x)

    out = out.view(-1, vocab_size)
    y = y.view(-1)
    
    loss = F.cross_entropy(out, y)
    print("train loss: ", loss.item())
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

train loss:  4.399728775024414
train loss:  3.921337127685547
train loss:  3.7772724628448486
train loss:  3.552271842956543
train loss:  3.532590627670288
train loss:  3.32916259765625
train loss:  3.3567519187927246
train loss:  3.3674685955047607
train loss:  3.271211862564087
train loss:  3.3747596740722656
train loss:  3.135763645172119
train loss:  3.4007458686828613
train loss:  3.318669319152832
train loss:  3.3493525981903076
train loss:  3.1340060234069824
train loss:  3.265143871307373
train loss:  3.3656771183013916
train loss:  3.1802759170532227
train loss:  3.1373696327209473
train loss:  3.160689353942871
train loss:  3.3638434410095215
train loss:  3.1838736534118652
train loss:  3.270162343978882
train loss:  3.2749247550964355
train loss:  3.113271474838257
train loss:  3.2191710472106934
train loss:  3.217006206512451
train loss:  3.195497512817383
train loss:  3.3132033348083496
train loss:  3.070438861846924
train loss:  3.2617454528808594
train loss:  3.061321496

In [374]:
torch.no_grad()
x_val, y_val = sample_batch(1000, train = False)
out_test = model(x_val)

print(out_test.shape)
print(y_val.shape)
val_loss = F.cross_entropy(out_test.view(-1, vocab_size), y_val.view(-1))
print("val loss: ", val_loss.item())






torch.Size([1000, 8, 65])
torch.Size([1000, 8])
val loss:  2.223334550857544


In [375]:
x =  torch.randn((8, 4, 2)) # B,T,C

attention = torch.tril(torch.ones(4,4)) # T,T 

attention[attention == 0] = float('-inf')
attention = F.softmax(attention, dim = -1)
wk = nn.Linear(2, 2)
wq = nn.Linear(2, 2)
wv = nn.Linear(2, 2)

k = wk(x)
q = wq(x)
v = wv(x)

print(k.shape)
print(q.shape)
print(v.shape)
print(k[0,:,:])
print(q[0,:,:])
print(v[0,:,:])

kq = q @ k.transpose(1,2)
kq = kq / torch.sqrt(torch.tensor(2.0))
kq = kq.masked_fill(attention == 0, float('-inf'))
kq = F.softmax(kq, dim = -1)

print(kq.shape)
print(v.shape)
print(kq[0,:,:])
print(v[0,:,:])
out = kq @ v
print(out[0,:,:])






torch.Size([8, 4, 2])
torch.Size([8, 4, 2])
torch.Size([8, 4, 2])
tensor([[-0.0265, -0.0258],
        [ 0.0693, -0.0883],
        [ 0.9197, -1.3583],
        [-0.4601,  0.1476]], grad_fn=<SliceBackward0>)
tensor([[ 0.1571, -0.6111],
        [ 0.0593, -0.5217],
        [-2.0064,  0.4112],
        [ 0.4170, -0.9944]], grad_fn=<SliceBackward0>)
tensor([[ 0.6799,  1.1357],
        [ 0.6579,  1.1188],
        [-1.3837, -0.6887],
        [ 0.4980,  0.9594]], grad_fn=<SliceBackward0>)
torch.Size([8, 4, 4])
torch.Size([8, 4, 2])
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.4932, 0.5068, 0.0000, 0.0000],
        [0.4915, 0.4213, 0.0871, 0.0000],
        [0.1606, 0.1726, 0.5417, 0.1251]], grad_fn=<SliceBackward0>)
tensor([[ 0.6799,  1.1357],
        [ 0.6579,  1.1188],
        [-1.3837, -0.6887],
        [ 0.4980,  0.9594]], grad_fn=<SliceBackward0>)
tensor([[ 0.6799,  1.1357],
        [ 0.6688,  1.1272],
        [ 0.4908,  0.9696],
        [-0.4646,  0.1224]], grad_fn=<SliceBackward0>)
